# Dataset 3 — Power Consumption of Tetouan City (UCI)  
**Situação**  
Uma equipe responsável pelo planejamento da distribuição de energia em Tétouan deseja identificar qual das três zonas apresenta o maior pico de consumo e observar as condições ambientais registradas nos momentos de maior demanda.

---

Grupo:
| Nome | RM |
| --- | --- |
| Caio Marinho Pereira | 572873|
<br>


## Workflow Orange
Workflow realizado no Orange para mineração de dados. Não foram encontrados valores ausentes. Foi gerado uma amostra de 15% dos dados.  
untitled.svg

## Import de Bibliotecas
Adiciona as bibliotecas de análise de dados que serão usadas.

In [2]:
import pandas as pd

## Carregar dados externos
Carrega a planilha com os dados, já tratados, e traduz o nome das colunas para o português.

In [3]:
dados = pd.read_csv('/content/tetuan_power.csv')

In [4]:
dados = dados.rename(columns = {'Temperature': 'Temperatura',
                                'Humidity': 'Humidade',
                                'Wind Speed': 'Velocidade_vento',
                                'Zone 1 Power Consumption': 'Consumo_Zona_1',
                                'Zone 2  Power Consumption': 'Consumo_Zona_2',
                                'Zone 3  Power Consumption': 'Consumo_Zona_3'})

### Maiores consumos registrados

Zona 1

In [5]:
max_consumo_1 = dados['Consumo_Zona_1'].max()
print(f"O maior consumo registrado é de: {max_consumo_1:.3f} kWh")

O maior consumo registrado é de: 51571.500 kWh


Zona 2

In [6]:
max_consumo_2 = dados['Consumo_Zona_2'].max()
print(f"O maior consumo registrado é de: {max_consumo_2:.3f} kWh")

O maior consumo registrado é de: 36437.200 kWh


Zona 3

In [7]:
max_consumo_3 = dados['Consumo_Zona_3'].max()
print(f"O maior consumo registrado é de: {max_consumo_3:.3f} kWh")

O maior consumo registrado é de: 47441.700 kWh


### Zona com maior pico

In [8]:
colunas_zonas = ['Consumo_Zona_1', 'Consumo_Zona_2', 'Consumo_Zona_3']

zona_maior_pico = dados[colunas_zonas].max().idxmax()
print(f"Zona com o maior pico: {zona_maior_pico}")


Zona com o maior pico: Consumo_Zona_1


### 70% do valor máximo
Define um limite superior para o Consumo da Zona 1, com base nos 30% valores mais altos

In [9]:
limite_70 = max_consumo_1 * .70
print(f"O limite de 70% do consumo máximo da zona 1 equivale a: {limite_70:.3f} kWh")

O limite de 70% do consumo máximo da zona 1 equivale a: 36100.050 kWh


DataFrame com registros de consumo acima desse limite

In [10]:
df_consumo_70 = dados[dados['Consumo_Zona_1'] > limite_70]
df_consumo_70['Consumo_Zona_1'].head()

,Consumo_Zona_1
3,42476.9
8,37286.7
12,40993.9
14,37475.2
19,40676.0


Quantidade e porcentagem de registros acima do limiar

In [11]:
print(f"A quantidade de registros acima do limite é de: {df_consumo_70.shape[0]}")
porcent_70 = (df_consumo_70.shape[0] / dados.shape[0]) * 100
print(f"Essa quantidade representa {porcent_70:.2f}% de registros do consumo total da zona 1")

A quantidade de registros acima do limite é de: 2388
Essa quantidade representa 30.37% de registros do consumo total da zona 1


### Temperatura Média


In [12]:
temp_media = dados['Temperatura'].mean()
print(f"{temp_media:.2f}°C")

18.79°C


Registros com temperatura acima da media e, simultaneamente, consumo alto da zona 1



In [13]:
df_temp_consumo = dados[(dados['Temperatura'] > temp_media) & (dados['Consumo_Zona_1'] > limite_70)]
df_temp_consumo[['Temperatura', 'Consumo_Zona_1']].head()

,Temperatura,Consumo_Zona_1
12,21.03,40993.9
14,27.06,37475.2
19,19.20,40676.0
21,27.11,40256.1
25,26.92,36511.9


### O que aconteceu com a quantidade de registros após a inclusão da condição ambiental

Com a inclusão da condição ambiental de Temperatura, ocorreu a:

* Redução na quantidade de dados (Filtragem Estrita): Ao adicionar a condição ambiental (Temperatura > temp_media), o filtro passa a utilizar o operador lógico E (&). Isso exige que ambas as condições sejam verdadeiras ao mesmo tempo para que uma linha seja mantida. Como resultado, a quantidade de registros no segundo DataFrame será menor ou igual à do primeiro.

* Eliminação de falsos positivos ambientais: O primeiro conjunto captura todos os picos de consumo, independentemente do motivo (como horários de pico comercial ou dias úteis). O segundo conjunto descarta os momentos de alto consumo que ocorreram em temperaturas amenas ou frias, isolando exclusivamente os picos energéticos correlacionados ao calor acima da média.